# 👗 AI Fashion Studio (Zara & H&M Lookbook Generator)
### **100% Free • Unlimited Generations • No Subscriptions • No Credit Limits**

This notebook runs the **State-of-the-Art Virtual Try-On AI (CatVTON & IDM-VTON)** on Google's free 16GB T4 GPU.

#### **How to use:**
1. In the top menu, go to **Runtime** > **Change runtime type** > Select **T4 GPU** (Free).
2. Run **Step 1** (Installs packages in ~40 seconds).
3. Run **Step 2** (Loads models and generates your public studio link).
4. Click the **`https://xxxx.gradio.live`** public link to upload clothes and create photos!

### Step 1: Install Dependencies (GPU-Safe & Complete)

In [ ]:
#@title Install AI Virtual Try-On Engine & Dependencies
!nvidia-smi
!rm -rf CatVTON
!git clone https://github.com/Zheng-Chong/CatVTON.git
%cd CatVTON

# Remove hardcoded torch/xformers pins so Colab's fast GPU PyTorch is preserved
!sed -i '/torch/d' requirements.txt
!sed -i '/xformers/d' requirements.txt

# Install core requirements and Detectron2/DensePose helpers (fvcore, iopath)
!pip install -q -r requirements.txt
!pip install -q fvcore iopath omegaconf diffusers transformers accelerate gradio huggingface_hub einops
print("\n✅ Environment Ready & All Modules Verified!")

### Step 2: Launch AI Fashion Studio Web App

In [ ]:
#@title Launch AI Fashion Studio Web App
import os
import torch
import gradio as gr
from PIL import Image
import numpy as np
from model.cloth_masker import AutoMasker
from model.pipeline import CatVTONPipeline
from utils import init_weight_dtype, resize_and_crop, resize_and_padding

# Initialize pipeline on free GPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

pipeline = CatVTONPipeline(
    base_ckpt="runwayml/stable-diffusion-inpainting",
    attn_ckpt="zhengchong/CatVTON",
    attn_ckpt_version="mix",
    weight_dtype=init_weight_dtype("fp16"),
    use_tf32=True,
    device=device
)
mask_processor = AutoMasker(densepose_path="zhengchong/DensePose", device=device)

def generate_zara_look(person_image, garment_image, cloth_type, num_steps, guidance_scale):
    if person_image is None or garment_image is None:
        return None, "Please upload both a model image and your garment!"
    
    person = resize_and_crop(person_image, (768, 1024))
    garment = resize_and_padding(garment_image, (768, 1024))
    
    # Auto-detect body and garment region
    mask = mask_processor(person, cloth_type=cloth_type)['mask']
    
    # Generate realistic drape and lighting
    result = pipeline(
        image=person,
        condition_image=garment,
        mask=mask,
        num_inference_steps=int(num_steps),
        guidance_scale=float(guidance_scale)
    )[0]
    
    return result, "Done! High-resolution Zara catalog photo ready."

# Build Zara/H&M Studio Dashboard
with gr.Blocks(theme=gr.themes.Monochrome(font=[gr.themes.GoogleFont("Inter"), "sans-serif"]), title="Zara / H&M AI Fashion Studio") as demo:
    gr.Markdown("""
    # 👗 ZARA & H&M STYLE AI FASHION STUDIO
    ### Private Brand Owner Edition • Unlimited Generations • 0 Subscriptions
    """)
    
    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown("### 1. Select / Upload Model")
            person_input = gr.Image(type="pil", label="Model (Studio pose, plain background)")
            cloth_type = gr.Radio(
                choices=["upper", "lower", "overall"],
                value="upper",
                label="Clothing Category (Tops, Pants/Skirts, or Dresses/Jumpsuits)"
            )
            
        with gr.Column(scale=1):
            gr.Markdown("### 2. Upload Your Product")
            garment_input = gr.Image(type="pil", label="Garment Photo (Flat-lay, hanger, or mannequin)")
            
            with gr.Accordion("Advanced Settings (Optional)", open=False):
                num_steps = gr.Slider(minimum=20, maximum=50, value=30, step=1, label="Inference Steps (Quality)")
                guidance_scale = gr.Slider(minimum=1.5, maximum=5.0, value=2.5, step=0.1, label="Clothing Fidelity / Guidance")
            
            btn_generate = gr.Button("✨ Generate Zara Model Photo", variant="primary", size="lg")
            
        with gr.Column(scale=1):
            gr.Markdown("### 3. Catalog-Ready Result")
            result_output = gr.Image(type="pil", label="Final Model Photo (Ready to download)")
            status_output = gr.Textbox(label="Status", interactive=False)

    btn_generate.click(
        fn=generate_zara_look,
        inputs=[person_input, garment_input, cloth_type, num_steps, guidance_scale],
        outputs=[result_output, status_output]
    )

# share=True creates a public live link for your MacBook/phone without exposing your local network
demo.launch(share=True, debug=True)
